# 02 · Kaggle: whisper-small on all JML transcribed data

Fine-tunes `openai/whisper-small` on the 8 transcribed JML archives (~134 h, 650 speakers)
and compares the result with finanfa-fon-asr-v6 on the same test speakers.

**Kaggle setup (once)**
1. *Settings → Accelerator*: **GPU T4 x2** (this notebook uses one T4). *Internet*: **on**.
2. *Add-ons → Secrets*: add `HF_TOKEN`, a Hugging Face token with **write** access, and attach it to this notebook.
   Checkpoints are pushed to a private repo on your account, so training can resume in the next session.
3. Run with **Save Version → Save & Run All (Commit)**: it runs in the background for up to 12 h,
   no need to keep the browser open.

**Sessions.** Kaggle stops a session after 12 h. The notebook stops training itself after
`TIME_BUDGET_HOURS`, pushes a checkpoint, and the next run of the same notebook resumes from it.
Repeat until it reports `MAX_STEPS` reached; the final test and finanfa comparison then run.
Keep `ARCHIVES` and the split settings the same across sessions, or the splits change.

In [ ]:
import os
import time
SESSION_START = time.time()
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one T4; Trainer would otherwise wrap the model in DataParallel

from pathlib import Path

# ---- Configuration ----
BASE_MODEL = "openai/whisper-small"
HUB_REPO_NAME = "kanbior-fon-asr-v1"  # private model repo on your HF account
ARCHIVES = range(8)          # audio_transcribed_000..007 = all transcribed audio (~15.5 GB)
MAX_STEPS = 12_000           # total across sessions, ~3.7 passes over the training set
BATCH_SIZE = 16
LEARNING_RATE = 1e-5
WARMUP_STEPS = 500
EVAL_SAVE_EVERY = 1_000      # validate, save and push a checkpoint every N steps
VAL_CLIPS = 300              # validation subset used during training (full test set at the end)
TIME_BUDGET_HOURS = 10.0     # from session start; leaves time to save and run the final evaluation
USE_HUB = True               # False: no pushing / resuming (single session only)

REPO_URL = "https://github.com/dyeto/kanbior.git"
JML_REPO = "JaimeMaLangue/JML-fon-speech-v1"
FINANFA_REPO = "finanfa-tech/finanfa-fon-asr-v6"

WORK = Path("/kaggle/working")
DATA_DIR = Path("/tmp/jml")           # extracted audio (not kept in the notebook output)
SPLITS_DIR = WORK / "splits"
OUTPUT_DIR = WORK / "whisper-small-fon"
RESUME_DIR = Path("/tmp/resume")    # kept outside OUTPUT_DIR so pushes never re-upload it

In [ ]:
!pip install -q "transformers>=5.24,<6" "accelerate>=1.16" "jiwer>=4.0" "faster-whisper==1.2.1"

In [ ]:
import json
import shutil
import subprocess
import sys
import time

import jiwer
import numpy as np
import soundfile as sf
import torch
import transformers
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback,
    WhisperForConditionalGeneration,
    WhisperProcessor,
)

print("torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (enable the GPU accelerator)")

In [ ]:
# Code from the kanbior repo: prepare_data.py (text cleaning, speaker splits, WER normalization)
CODE_DIR = Path("/tmp/kanbior")
if not CODE_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CODE_DIR)], check=True)
sys.path.insert(0, str(CODE_DIR / "kanbior-asr"))
from prepare_data import normalize_for_wer, read_split

In [ ]:
if USE_HUB:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    api = HfApi()
    HUB_MODEL_ID = f"{api.whoami()['name']}/{HUB_REPO_NAME}"
    api.create_repo(HUB_MODEL_ID, private=True, exist_ok=True)
    print("Checkpoints go to https://huggingface.co/" + HUB_MODEL_ID)

## Data: download, extract, split

In [ ]:
def fetch_archive(i):
    """Download one JML archive, extract it into DATA_DIR, delete the tar (keeps disk use low)."""
    name = f"audio_transcribed_{i:03d}.tar"
    marker = DATA_DIR / f".{name}.done"
    if marker.exists():
        return
    free_gb = shutil.disk_usage(DATA_DIR).free / 1e9
    assert free_gb > 5, f"Only {free_gb:.1f} GB free; need ~5 GB per archive"
    t = time.time()
    tar = hf_hub_download(JML_REPO, name, repo_type="dataset", local_dir=DATA_DIR / "_download")
    subprocess.run(["tar", "-xf", tar, "-C", str(DATA_DIR)], check=True)
    os.remove(tar)
    marker.touch()
    print(f"{name}: {time.time() - t:.0f} s, {shutil.disk_usage(DATA_DIR).free / 1e9:.0f} GB free")

DATA_DIR.mkdir(parents=True, exist_ok=True)
hf_hub_download(JML_REPO, "transcribed.csv", repo_type="dataset", local_dir=DATA_DIR)
for i in ARCHIVES:
    fetch_archive(i)

In [ ]:
out = subprocess.run(
    [sys.executable, str(CODE_DIR / "kanbior-asr" / "prepare_data.py"),
     "--data-dir", str(DATA_DIR), "--out-dir", str(SPLITS_DIR)],
    check=True, capture_output=True, text=True,
)
print(out.stdout)

In [ ]:
def take(rows, n, seed=0):
    if n is None or n >= len(rows):
        return rows
    idx = np.random.default_rng(seed).choice(len(rows), size=n, replace=False)
    return [rows[i] for i in sorted(idx)]

train_rows = read_split(SPLITS_DIR / "train.csv")
val_rows = take(read_split(SPLITS_DIR / "val.csv"), VAL_CLIPS)
test_rows = read_split(SPLITS_DIR / "test.csv")
for name, rows in [("train", train_rows), ("val", val_rows), ("test", test_rows)]:
    print(f"{name:5s}: {len(rows):6,} clips, {sum(r['duration'] for r in rows) / 3600:6.1f} h, "
          f"{len({r['speaker_id'] for r in rows})} speakers")

## Model and data pipeline

In [ ]:
processor = WhisperProcessor.from_pretrained(BASE_MODEL, language="french", task="transcribe")
tokenizer = processor.tokenizer

model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL)
model.generation_config.language = "fr"   # Whisper has no Fon token; reuse French like finanfa
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.use_cache = False            # required with gradient checkpointing


class FonSpeechDataset(torch.utils.data.Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        row = self.rows[i]
        audio, sr = sf.read(DATA_DIR / row["audio_path"], dtype="float32")
        assert sr == 16000, f"{row['id']}: expected 16 kHz, got {sr}"
        return {"audio": audio, "text": row["text"]}


class Collator:
    def __init__(self, processor, decoder_start_token_id):
        self.processor = processor
        self.decoder_start_token_id = decoder_start_token_id

    def __call__(self, batch):
        features = self.processor.feature_extractor(
            [b["audio"] for b in batch], sampling_rate=16000, return_tensors="pt"
        )
        labels = self.processor.tokenizer([b["text"] for b in batch], padding=True, return_tensors="pt")
        label_ids = labels.input_ids.masked_fill(labels.attention_mask.ne(1), -100)
        if (label_ids[:, 0] == self.decoder_start_token_id).all():
            label_ids = label_ids[:, 1:]
        return {"input_features": features.input_features, "labels": label_ids}


train_ds, val_ds, test_ds = (FonSpeechDataset(r) for r in (train_rows, val_rows, test_rows))
collator = Collator(processor, model.config.decoder_start_token_id)

In [ ]:
def score(refs, preds):
    out = {}
    for suffix, keep_tones in (("", True), ("_no_tones", False)):
        r = [normalize_for_wer(t, keep_tones) for t in refs]
        p = [normalize_for_wer(t, keep_tones) for t in preds]
        out["wer" + suffix] = 100 * jiwer.wer(r, p)
        out["cer" + suffix] = 100 * jiwer.cer(r, p)
    return out


def decode(ids):
    ids = np.where(ids < 0, tokenizer.pad_token_id, ids)
    return [t.strip() for t in tokenizer.batch_decode(ids, skip_special_tokens=True)]


def compute_metrics(eval_pred):
    return score(decode(eval_pred.label_ids), decode(eval_pred.predictions))


class TimeBudget(TrainerCallback):
    """Stop (and save) before Kaggle's session limit; the next session resumes."""
    def __init__(self, hours, start):
        self.deadline = start + hours * 3600

    def on_step_end(self, args, state, control, **kwargs):
        if time.time() > self.deadline:
            print(f"Time budget reached at step {state.global_step}; saving and stopping.")
            control.should_save = True
            control.should_training_stop = True
        return control

## Train (resumes from the last pushed checkpoint)

In [ ]:
resume_from = None
if USE_HUB and any(f.startswith("last-checkpoint/") for f in api.list_repo_files(HUB_MODEL_ID)):
    snapshot_download(HUB_MODEL_ID, allow_patterns="last-checkpoint/*", local_dir=RESUME_DIR)
    resume_from = str(RESUME_DIR / "last-checkpoint")
    done = json.load(open(Path(resume_from) / "trainer_state.json"))["global_step"]
    print(f"Resuming from step {done:,} of {MAX_STEPS:,}")
else:
    print("Starting a new run")

args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    max_steps=MAX_STEPS,
    activation_checkpointing=torch.cuda.is_available(),
    fp16=torch.cuda.is_available(),
    eval_strategy="steps",
    eval_steps=EVAL_SAVE_EVERY,
    save_steps=EVAL_SAVE_EVERY,
    save_total_limit=2,
    logging_steps=50,
    predict_with_generate=True,
    generation_max_length=225,
    remove_unused_columns=False,
    dataloader_num_workers=4 if torch.cuda.is_available() else 0,
    report_to="none",
    push_to_hub=USE_HUB,
    hub_model_id=HUB_MODEL_ID if USE_HUB else None,
    hub_strategy="checkpoint",   # also pushes the latest checkpoint to last-checkpoint/ for resuming
    hub_private_repo=True,
    hub_always_push=True,        # don't skip the final save if a previous upload is still running
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
    callbacks=[TimeBudget(TIME_BUDGET_HOURS, SESSION_START)],
)

start = time.time()
trainer.train(resume_from_checkpoint=resume_from)
FINISHED = trainer.state.global_step >= MAX_STEPS
print(f"Session trained for {(time.time() - start) / 3600:.1f} h, now at step "
      f"{trainer.state.global_step:,}/{MAX_STEPS:,}" + ("" if FINISHED else " -> run the notebook again to continue"))

In [ ]:
# Loss and validation WER over the whole run (all sessions)
for log in trainer.state.log_history:
    if "eval_wer" in log:
        print(f"step {log['step']:6,}  val WER {log['eval_wer']:5.1f}%  no tones {log['eval_wer_no_tones']:5.1f}%"
              f"  CER {log['eval_cer']:5.1f}%")
    elif "loss" in log and log["step"] % 500 == 0:
        print(f"step {log['step']:6,}  train loss {log['loss']:.3f}")

## Final evaluation (only when training is finished)

Full test set: speakers never seen in training or validation.

In [ ]:
if FINISHED:
    pred = trainer.predict(test_ds, metric_key_prefix="test")
    ours = decode(pred.predictions)
    refs = [r["text"] for r in test_rows]
    ours_metrics = score(refs, ours)
    print("kanbior whisper-small:", {k: round(v, 1) for k, v in ours_metrics.items()})
    for r, p in list(zip(test_rows, ours))[:5]:
        print(f"\n{r['id']} ({r['speaker_id']})\n  ref: {r['text']}\n  hyp: {p}")
else:
    print("Training not finished yet; skipping the test set.")

In [ ]:
# finanfa-fon-asr-v6 on the same test clips (faster-whisper; GPU if CTranslate2 can use it, else a CPU subset)
if FINISHED:
    from faster_whisper import WhisperModel

    finanfa_dir = snapshot_download(FINANFA_REPO)
    try:
        fw = WhisperModel(finanfa_dir, device="cuda", compute_type="float16")
        subset = list(range(len(test_rows)))
    except Exception as e:
        print("CTranslate2 GPU unavailable, using CPU on 500 clips:", e)
        fw = WhisperModel(finanfa_dir, device="cpu", compute_type="int8")
        subset = sorted(np.random.default_rng(0).choice(len(test_rows), size=min(500, len(test_rows)), replace=False))

    finanfa_preds = []
    t = time.time()
    for i in subset:
        segments, _ = fw.transcribe(str(DATA_DIR / test_rows[i]["audio_path"]), language="fr", beam_size=1)
        finanfa_preds.append(" ".join(s.text.strip() for s in segments))
    print(f"finanfa transcribed {len(subset):,} clips in {(time.time() - t) / 60:.0f} min")

    sub_refs = [refs[i] for i in subset]
    comparison = {
        "clips": len(subset),
        "kanbior": score(sub_refs, [ours[i] for i in subset]),
        "finanfa-v6": score(sub_refs, finanfa_preds),
    }
    print(f"{'':12s} {'WER':>6s} {'WER no tones':>13s} {'CER':>6s}")
    for name in ("kanbior", "finanfa-v6"):
        m = comparison[name]
        print(f"{name:12s} {m['wer']:6.1f} {m['wer_no_tones']:13.1f} {m['cer']:6.1f}")
    print("Note: finanfa may have trained on some of these speakers, which would favour it.")

In [ ]:
if FINISHED:
    results = {"steps": trainer.state.global_step, "train_hours": sum(r["duration"] for r in train_rows) / 3600,
               "test": ours_metrics, "comparison": comparison}
    (WORK / "results.json").write_text(json.dumps(results, indent=2))
    final_dir = OUTPUT_DIR / "final"
    trainer.save_model(str(final_dir))   # also saves the processor
    if USE_HUB:
        trainer.push_to_hub(commit_message=f"Final model, test WER {ours_metrics['wer']:.1f}%")
        print("Final model: https://huggingface.co/" + HUB_MODEL_ID)